# Modelos preentrenados : YOLO  

## Objetivos académicos  
1. Comprender la utilidad de los **modelos preentrenados** en tareas de visión artificial y cómo aprovecharlos mediante *transfer learning*.  
2. Conocer la **intuición y funcionamiento** detrás de arquitecturas populares como **ResNet** y **YOLO**, aplicadas a la clasificación y detección de objetos en imágenes.  


## Uso de modelos entrenados y transfer learning  
En el campo de la visión artificial, los **modelos preentrenados** son redes neuronales que ya han sido entrenadas con grandes conjuntos de datos (como *ImageNet* o *COCO*), aprendiendo patrones visuales generales: bordes, texturas, formas y objetos.  
El **transfer learning** consiste en reutilizar ese conocimiento previo y adaptarlo a una nueva tarea, evitando entrenar desde cero y reduciendo el tiempo y los recursos necesarios.  
En lugar de que la red "aprenda a ver" desde el principio, aprovechamos una base ya consolidada y solo ajustamos las últimas capas para especializarla en nuestro problema específico.  




### ResNet  
**ResNet (Residual Network)** 🧠 es un modelo de clasificación de imágenes que introdujo el concepto de **conexiones residuales**.  
Estas conexiones permiten que la información fluya sin perderse entre muchas capas profundas, resolviendo el problema del *desvanecimiento del gradiente*.  
Intuitivamente, una ResNet aprende a "corregir" los errores de la predicción anterior en lugar de recalcularlo todo desde cero.  
Esto le permite alcanzar gran profundidad (más de 100 capas) sin perder capacidad de aprendizaje, y se usa ampliamente como base para tareas de reconocimiento y extracción de características visuales.  

In [ ]:
#!/bin/bash
!curl -L -o fer2013.zip https://www.kaggle.com/api/v1/datasets/download/msambare/fer2013

In [ ]:
import zipfile
import os 
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

In [ ]:
destination_path=os.path.join(os.getcwd(),'fer2013') 
with zipfile.ZipFile("fer2013.zip", 'r') as zip_ref:
    zip_ref.extractall(destination_path)


In [ ]:
train_dir = "data/fer2013/train"
test_dir = "data/fer2013/test"

In [ ]:
# Aumentación y normalización
datagen_train = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

datagen_test = ImageDataGenerator(rescale=1./255)

train_gen = datagen_train.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32
)

test_gen = datagen_test.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    color_mode="rgb",
    class_mode="categorical",
    batch_size=32,
    shuffle=False
)

In [ ]:

# Cargar modelo base preentrenado
base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False  # congelar capas

# Construir la red
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

# Compilar
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# ---------------------------------------------------------
# 5️⃣ Entrenamiento
# ---------------------------------------------------------
history = model.fit(
    train_gen,
    validation_data=test_gen,
    epochs=10
)


In [ ]:
# ---------------------------------------------------------
# 6️⃣ Visualización del desempeño
# ---------------------------------------------------------
import matplotlib.pyplot as plt

plt.figure(figsize=(8,4))
plt.plot(history.history['accuracy'], label='Entrenamiento')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Evolución de la exactitud 📈')
plt.xlabel('Épocas')
plt.ylabel('Exactitud')
plt.legend()
plt.show()

In [ ]:
# =========================================================
# 🎯 Prueba visual: Predicción de una imagen aleatoria
# =========================================================

# 🔹 Seleccionar un índice aleatorio del conjunto de test
idx = random.randint(0, len(test_gen.filenames) - 1)

# 🔹 Cargar la imagen y etiqueta real
img_path = test_gen.filepaths[idx]
true_label = test_gen.classes[idx]
class_names = list(test_gen.class_indices.keys())

# 🔹 Preparar la imagen para el modelo
img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
img_array = tf.keras.preprocessing.image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

# 🔹 Realizar predicción
preds = model.predict(img_array)
pred_label = np.argmax(preds)
pred_class = class_names[pred_label]
true_class = class_names[true_label]

# 🔹 Visualizar resultado
plt.figure(figsize=(4, 4))
plt.imshow(plt.imread(img_path))
color = "green" if pred_class == true_class else "red"
plt.title(f"Verdadero: {true_class}\nPredicción: {pred_class}", color=color, fontsize=12, weight="bold")
plt.axis("off")
plt.show()




### YOLO  
**YOLO (You Only Look Once)** 🚀 es un modelo de **detección y segmentación de objetos** que destaca por su velocidad y precisión.  
A diferencia de otros enfoques que analizan una imagen por regiones, YOLO la procesa **de una sola vez**, dividiéndola en una cuadrícula y prediciendo simultáneamente **qué objetos hay y dónde están**.  
Cada celda de la cuadrícula genera una serie de *bounding boxes* (cajas delimitadoras) junto con su probabilidad de pertenecer a una clase específica.  
Esta estrategia hace que YOLO sea extremadamente rápido —ideal para aplicaciones en tiempo real como conducción autónoma o videovigilancia— y fácil de adaptar mediante *transfer learning* a nuevos conjuntos de objetos o contextos. 👁️📦  


#### Modelo

In [ ]:
! pip install -q ultralytics gradio | echo "Instalado"


In [ ]:
import os
from typing import Tuple
import numpy as np
from PIL import Image
import gradio as gr
from ultralytics import YOLO

In [ ]:


# -----------------------------
# Carga del modelo (una sola vez)
# -----------------------------
# Puedes cambiar a "yolov8s.pt", "yolov8m.pt", etc. según precisión/velocidad.
MODEL_NAME = os.getenv("YOLO_MODEL", "yolov8n.pt")
model = YOLO(MODEL_NAME)  # descarga automática si no está presente

def detectar(imagen: np.ndarray, conf: float, iou: float, max_det: int, imgsz: int) -> np.ndarray:
    """
    Ejecuta la predicción sobre una imagen (numpy array RGB) y devuelve la imagen anotada.
    """
    # Ultralytics acepta numpy/PIL directamente
    results = model.predict(
        source=imagen,
        conf=conf,
        iou=iou,
        max_det=max_det,
        imgsz=int(imgsz),
        verbose=False
    )

    # results[0].plot() devuelve un array BGR; lo convertimos a RGB para Gradio
    bgr_annotated = results[0].plot()
    rgb_annotated = bgr_annotated[:, :, ::-1]  # BGR -> RGB
    return rgb_annotated

# -----------------------------
# Interfaz de Gradio
# -----------------------------
with gr.Blocks(title="Detección con YOLO (Ultralytics)") as demo:
    gr.Markdown(
        """
        # Detección de objetos con YOLO
        Sube una imagen y el modelo dibujará cajas y etiquetas sobre los objetos detectados.
        Puedes ajustar el umbral de confianza, IoU, tamaño de imagen y máximo de detecciones.
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            input_image = gr.Image(type="numpy", label="Imagen de entrada", sources=["upload", "clipboard"])
            conf_slider = gr.Slider(0.1, 0.95, value=0.5, step=0.05, label="Confianza mínima (conf)")
            iou_slider  = gr.Slider(0.1, 0.95, value=0.45, step=0.05, label="IoU (NMS)")
            imgsz_slider = gr.Slider(320, 1280, value=640, step=32, label="Tamaño de imagen (imgsz)")
            maxdet_slider = gr.Slider(10, 300, value=100, step=10, label="Máximo de detecciones (max_det)")
            btn = gr.Button("Detectar")
        with gr.Column(scale=1):
            output_image = gr.Image(type="numpy", label="Imagen anotada")

    # Acciones
    btn.click(
        fn=detectar,
        inputs=[input_image, conf_slider, iou_slider, maxdet_slider, imgsz_slider],
        outputs=output_image
    )

    # También ejecutar automáticamente al cambiar la imagen
    input_image.change(
        fn=detectar,
        inputs=[input_image, conf_slider, iou_slider, maxdet_slider, imgsz_slider],
        outputs=output_image
    )


demo.launch()